In [ ]:

import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import numpy as np

# Objective: Select the best scientists for a specific team.
# Use Tensorflow for model development.

# 1. Data Preparation (Mock Data)
data = {
    'experience_years': np.random.rand(100) * 20,
    'publications_count': np.random.rand(100) * 50,
    'collaboration_score': np.random.randint(1, 6, 100),
    'project_success_rate': np.random.rand(100),
    'hourly_rate': np.random.randint(20, 100, 100),
    'team_size': np.random.randint(2, 10, 100),
    'location': np.random.choice(['Remote', 'On-site'], 100),
    'on_time_completion': np.random.randint(0, 3, 100), # 0: Low, 1: Medium, 2: High Fit
    'overtime_pay': np.random.randint(0, 3, 100), # 0: Low, 1: Medium, 2: High Fit
    'team_fit_label': np.random.randint(0, 3, 100) # 0: Low, 1: Medium, 2: High Fit
}

df = pd.DataFrame(data)

scientists_df = pd.DataFrame(data)

scientists_df['value_score'] = (scientists_df['experience_years'] * scientists_df['project_success_rate']) / scientists_df['hourly_rate']
display(scientists_df.head())

# Define features and labels
numerical_features = ['experience_years', 'publications_count', 'collaboration_score', 'project_success_rate','hourly_rate','team_size','on_time_completion','overtime_pay']
categorical_features = ['location']

X = df[numerical_features + categorical_features]
y = df['team_fit_label'].values

# Create a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Apply preprocessing
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

# Convert to TensorFlow datasets
# This is optional but good practice for large datasets
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train)).batch(10).shuffle(len(X_train))
test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(10).shuffle(len(X_test))

# 2. Build the TensorFlow Keras Model
model = tf.keras.Sequential([
    # Input layer requires an input shape
    tf.keras.layers.Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    tf.keras.layers.Dropout(0.2), # Dropout for regularization
    tf.keras.layers.Dense(64, activation='relu'),
    # Output layer with 3 units (for 3 classes: Low, Medium, High) and softmax activation
    tf.keras.layers.Dense(3, activation='softmax')
])

# 3. Compile the Model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy', # Use this for integer labels
    metrics=['accuracy']
)

# 4. Train the Model
print("Training the model...")
history = model.fit(
    train_dataset,
    epochs=10, # Number of passes over the training data
    validation_data=test_dataset
)

# 5. Evaluate the Model
test_loss, test_accuracy = model.evaluate(test_dataset)
print(f"\nTest Accuracy: {test_accuracy*100:.2f}%")

# 6. Make Predictions for New Candidates
# Example new candidate data (standardized using the same scaler)
# Ensure new_candidates has the same columns as X used for training
new_candidates_data = pd.DataFrame({
    'experience_years': [10, 2, 6],
    'publications_count': [25, 1, 12],
    'collaboration_score': [4, 2, 5],
    'project_success_rate': [0.8, 0.1, 0.5],
    'hourly_rate': [70, 30, 90],
    'team_size': [5, 3, 8],
    'on_time_completion': [1, 0, 2],
    'overtime_pay': [2, 0, 1],
    'value_score': [0.8, 0.1, 0.5],
    'location': ['Remote', 'On-site','Remote']
})

new_candidates_scaled = preprocessor.transform(new_candidates_data)

predictions = model.predict(new_candidates_scaled)
predicted_classes = np.argmax(predictions, axis=1)

print("\nPredictions for new candidates:")
for i, prediction in enumerate(predicted_classes):
    if prediction == 0:
        fit_label = "Low Fit"
    elif prediction == 1:
        fit_label = "Medium Fit"
    else:
        fit_label = "High Fit"
    print(f"Candidate {i+1}: {fit_label} (Confidence: {predictions[i][prediction]*100:.2f}%)")

###

,experience_years,publications_count,collaboration_score,project_success_rate,hourly_rate,team_size,location,on_time_completion,overtime_pay,team_fit_label,value_score
0,14.189591,4.949370,2,0.971137,83,6,On-site,1,0,0,0.166025
1,4.610334,47.378038,3,0.436734,84,9,Remote,0,0,0,0.023970
2,6.906545,1.903369,4,0.282268,46,9,On-site,0,0,2,0.042380
3,4.073130,10.923372,1,0.424680,25,6,Remote,1,2,2,0.069191
4,6.348942,36.732222,3,0.962013,92,6,On-site,1,1,0,0.066389


Training the model...
Epoch 1/10


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.4334 - loss: 1.0926 - val_accuracy: 0.4000 - val_loss: 1.0849
Epoch 2/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4246 - loss: 1.0721 - val_accuracy: 0.3500 - val_loss: 1.0827
Epoch 3/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4699 - loss: 1.0256 - val_accuracy: 0.4000 - val_loss: 1.0913
Epoch 4/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5273 - loss: 0.9739 - val_accuracy: 0.4500 - val_loss: 1.0896
Epoch 5/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4700 - loss: 0.9698 - val_accuracy: 0.4500 - val_loss: 1.0908
Epoch 6/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5847 - loss: 0.9622 - val_accuracy: 0.4000 - val_loss: 1.0995
Epoch 7/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5835 - loss: 0.9524 - val_accuracy: 0.4000 - val_loss: 1.0998
Epoch 8/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5860 - loss: 0.9000 - val_accuracy: 0.4500 - val_loss: 1.1062
Epoch 9/10
8/8 ━━━


Using TensorFlow for scientist team selection in a pharmaceutical firm involves creating a predictive model based on historical data of successful and unsuccessful teams/scientists. The process generally requires significant data preparation and careful ethical considerations to avoid bias.
The code below outlines a general structure for a multi-class classification model using Keras within TensorFlow to predict the potential success or "fit" of a scientist candidate for a specific team, based on various features.
Python Code for Scientist Team Selection Model
This example assumes you have pre-processed data in a Pandas DataFrame and converted it into a TensorFlow-compatible format.

Key Considerations
Data Quality is Paramount: The model's effectiveness depends entirely on the quality and relevance of your historical HR and performance data. Ensure data is clean, comprehensive, and unbiased.
Ethical AI in HR: Using AI for HR decisions carries a significant risk of perpetuating existing biases present in historical data. Implement rigorous fairness and bias checks.
Interpretability: For sensitive HR decisions, models with higher interpretability (like logistic regression or decision forests) might be preferred over a deep neural network, as they provide clearer, easier-to-understand explanations for their predictions.
Domain Expertise: Involve HR professionals and scientific leads to define the criteria for a "successful" scientist or team, ensuring the model aligns with the firm's strategic goals.
